In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

pd.set_option('display.max_columns', None)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from DATA.TOOLS.fetchPlayersStats import FetchPlayersStats
from DATA.TOOLS.fetchTeamStats import *

# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
# Add the parent directory to sys.path
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from DATA.TOOLS.playerPositions import *

Features I plan on adding in the future

Player-specific usage and scoring data

- /BoxScoreScoringV2: Breaks down how a player scores (paint, midrange, 3s, free throws). *

Shot quality and location data

- /ShotChartDetail: Individual shot attempts, zones, and frequencies.

- /LeagueDashPlayerShotLocations: Aggregated shot location tendencies.

- /LeagueDashPlayerPtShot: Breaks down shooting by play type and situation.

Opponent and matchup data

- /LeagueDashPtDefend: How defenders contest shots and limit scoring. *

- /LeagueDashTeamStats with defense filters: Opponent’s defensive efficiency.

- /BoxScoreMatchupsV3: Player-vs-player defensive assignments.

Game context and pace

- /ScoreboardV2 or /PlayByPlayV2: For back-to-backs, rest, or pace indicators.

- /TeamDashboardByGeneralSplits: Team-level pace, offensive rating, and context.


In [ ]:
pd.set_option('display.max_columns', None)
def convert_min_to_float(min_str):
    try:
        if isinstance(min_str, str) and ":" in min_str:
            minutes, seconds = map(int, min_str.split(":"))
            total_minutes = minutes + seconds / 60
            return round(total_minutes, 2)
        elif isinstance(min_str, (int, float)):
            return float(min_str)
        else:
            return 0
    except:
        return 0
    

s19_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
s19_regular['IS_PLAYOFF'] = 0
s19_regular['MIN'] = s19_regular['MIN'].apply(convert_min_to_float)

s20_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S20.csv')
s20_regular['IS_PLAYOFF'] = 0
s20_regular['MIN'] = s20_regular['MIN'].apply(convert_min_to_float)

s21_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S21.csv')
s21_regular['IS_PLAYOFF'] = 0
s21_regular['MIN'] = s21_regular['MIN'].apply(convert_min_to_float)

s22_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S22.csv')
s22_regular['IS_PLAYOFF'] = 0
s22_regular['MIN'] = s22_regular['MIN'].apply(convert_min_to_float)

s23_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S23.csv')
s23_regular['IS_PLAYOFF'] = 0
s23_regular['MIN'] = s23_regular['MIN'].apply(convert_min_to_float)

s24_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S24.csv')
s24_regular['IS_PLAYOFF'] = 0
s24_regular['MIN'] = s24_regular['MIN'].apply(convert_min_to_float)

s25_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S25.csv')
s25_regular['IS_PLAYOFF'] = 0
s25_regular['MIN'] = s25_regular['MIN'].apply(convert_min_to_float)

In [ ]:
from nba_api.stats.endpoints import boxscorematchupsv3
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
from requests.exceptions import ReadTimeout, ConnectionError

def fetch_matchup_data(gameId, sleep_time=1.0):
    """Fetch matchup data for a single game with error handling"""
    try:
        time.sleep(sleep_time)
        
        boxscore = boxscorematchupsv3.BoxScoreMatchupsV3(
            game_id=f'00{gameId}',
            timeout=60
        )
        data_frames = boxscore.get_data_frames()
        if data_frames and len(data_frames) > 0 and not data_frames[0].empty:
            # Process the matchup data similar to your original logic
            boxscore_df = data_frames[0]
            boxscore_df['matchupMinutes'] = round(boxscore_df['matchupMinutesSort'] / 60, 2) 
            
            def_df = (
                boxscore_df.groupby('personIdDef')
                .agg({
                    'gameId': 'first',
                    'teamId': 'first',
                    'matchupFieldGoalsMade': 'sum',
                    'matchupFieldGoalsAttempted': 'sum',
                    'matchupThreePointersMade': 'sum',
                    'matchupThreePointersAttempted': 'sum',
                    'matchupTurnovers': 'sum',
                    'matchupBlocks': 'sum',
                    'shootingFouls': 'sum',
                    'matchupAssists': 'sum',
                    'playerPoints': 'sum',
                    'matchupMinutes': 'sum'
                })
                .reset_index()
            )
            def_df['DEF_FG_PCT_ALLOWED'] = round(def_df['matchupFieldGoalsMade'] / def_df['matchupFieldGoalsAttempted'], 3)
            def_df['DEF_3PT_PCT_ALLOWED'] = round(def_df['matchupThreePointersMade'] / def_df['matchupThreePointersAttempted'], 3)
            def_df['PTS_ALLOWED_PER_MIN'] = round(def_df['playerPoints'] / def_df['matchupMinutes'], 2)
            def_df['DEF_TOV_FORCED_PER_MIN'] = def_df['matchupTurnovers'] / def_df['matchupMinutes']
            def_df['DEF_BLOCKS_PER_MIN'] = def_df['matchupBlocks'] / def_df['matchupMinutes']
            def_df['DEF_SHOOTING_FOULS_PER_MIN'] = def_df['shootingFouls'] / def_df['matchupMinutes']
            def_df['DEF_AST_ALLOWED_PER_MIN'] = def_df['matchupAssists'] / def_df['matchupMinutes']
            
            return def_df, gameId, None
        else:
            return None, gameId, "No data available"
            
    except ReadTimeout:
        return None, gameId, "Timeout"
    except Exception as e:
        return None, gameId, str(e)

# Get unique game IDs
gameIds = s25_regular['GAME_ID'].unique()
gameLogs = []
failed_games = []

print(f"Processing {len(gameIds)} games using ThreadPoolExecutor for matchup data...")

max_workers = 3 
sleep_time = 1.5  

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_game = {
        executor.submit(fetch_matchup_data, gameId, sleep_time): gameId 
        for gameId in gameIds
    }
    
    # Process completed tasks
    for i, future in enumerate(as_completed(future_to_game), 1):
        gameId = future_to_game[future]
        
        try:
            result, game_id, error = future.result()
            
            if result is not None:
                gameLogs.append(result)
                print(f"✓ Game {gameId} completed ({i}/{len(gameIds)})")
            else:
                failed_games.append((gameId, error))
                print(f"✗ Game {gameId} failed: {error} ({i}/{len(gameIds)})")
                
        except Exception as e:
            failed_games.append((gameId, str(e)))
            print(f"✗ Game {gameId} exception: {str(e)} ({i}/{len(gameIds)})")

print(f"\nCompleted!")
print(f"Successfully processed: {len(gameLogs)} games")
print(f"Failed games: {len(failed_games)}")

if failed_games:
    print(f"Failed games: {failed_games[:10]}...")

if gameLogs:
    gameLogsTotal = pd.concat(gameLogs, ignore_index=True)
    print(f"\nFinal dataset shape: {gameLogsTotal.shape}")
    gameLogsTotal.head()
else:
    print("No games were successfully processed!")

Processing 1230 games using ThreadPoolExecutor for matchup data...
✓ Game 22400061 completed (1/1230)
✓ Game 22400070 completed (2/1230)
✓ Game 22400062 completed (3/1230)
✓ Game 22400072 completed (4/1230)
✓ Game 22400071 completed (5/1230)
✓ Game 22400069 completed (6/1230)
✓ Game 22400065 completed (7/1230)
✓ Game 22400064 completed (8/1230)
✓ Game 22400066 completed (9/1230)
✓ Game 22400068 completed (10/1230)
✓ Game 22400067 completed (11/1230)
✓ Game 22400063 completed (12/1230)
✓ Game 22400076 completed (13/1230)
✓ Game 22400073 completed (14/1230)
✓ Game 22400074 completed (15/1230)
✓ Game 22400075 completed (16/1230)
✓ Game 22400080 completed (17/1230)
✓ Game 22400077 completed (18/1230)
✓ Game 22400084 completed (19/1230)
✓ Game 22400079 completed (20/1230)
✓ Game 22400082 completed (21/1230)
✓ Game 22400081 completed (22/1230)
✓ Game 22400086 completed (23/1230)
✓ Game 22400085 completed (24/1230)
✓ Game 22400078 completed (25/1230)
✓ Game 22400094 completed (26/1230)
✓ Game

In [100]:
gameLogsTotal.rename(columns={'personIdDef': 'PLAYER_ID', 'teamId': 'TEAM_ID', 'gameId': 'GAME_ID'}, inplace=True)
gameLogsTotal

,PLAYER_ID,GAME_ID,TEAM_ID,matchupFieldGoalsMade,matchupFieldGoalsAttempted,matchupThreePointersMade,matchupThreePointersAttempted,playerPoints,matchupMinutes,DEF_FG_PCT_ALLOWED,DEF_3PT_PCT_ALLOWED,PTS_ALLOWED_PER_MIN
0,201143,0022400061,1610612752,11,16,1,4,26,9.64,0.688,0.250,2.70
1,201950,0022400061,1610612752,8,13,3,6,21,12.03,0.615,0.500,1.75
2,1626157,0022400061,1610612738,11,19,6,11,28,9.00,0.579,0.545,3.11
3,1626166,0022400061,1610612738,4,5,2,3,10,8.10,0.800,0.667,1.23
4,1627759,0022400061,1610612752,5,10,0,2,12,10.85,0.500,0.000,1.11
...,...,...,...,...,...,...,...,...,...,...,...,...
26238,1642267,0022401190,1610612748,8,13,0,4,18,15.32,0.615,0.000,1.17
26239,1642273,0022401190,1610612748,4,11,2,5,10,5.94,0.364,0.400,1.68
26240,1642276,0022401190,1610612764,6,11,0,3,14,7.29,0.545,0.000,1.92
26241,1642352,0022401190,1610612764,1,9,1,4,8,10.67,0.111,0.250,0.75


In [103]:
def mergeMatchupDATA(data, matchup_data):
    data = data.copy()
    matchup_data = matchup_data.copy()
    

    if 'GAME_ID' in matchup_data.columns:
        matchup_data['GAME_ID'] = matchup_data['GAME_ID'].astype(int)
    
    # Ensure main data PLAYER_ID is also string
    if 'PLAYER_ID' in data.columns:
        data['PLAYER_ID'] = data['PLAYER_ID']
    
    merge_columns = [
        'GAME_ID', 'PLAYER_ID',
        'matchupFieldGoalsMade', 'matchupFieldGoalsAttempted',
        'matchupThreePointersMade', 'matchupThreePointersAttempted',
        'playerPoints', 'matchupMinutes', 'matchupFieldGoalsPercentage',
        'matchupThreePointersPercentage', 'DEF_FG_PCT_ALLOWED',
        'DEF_3PT_PCT_ALLOWED', 'PTS_ALLOWED_PER_MIN'
    ]
    
    available_columns = [col for col in merge_columns if col in matchup_data.columns]
    matchup_subset = matchup_data[available_columns].copy()
    
    merged_data = data.merge(
        matchup_subset,
        on=['GAME_ID', 'PLAYER_ID'],
        how='left',
        suffixes=('', '_matchup')
    )
    return merged_data

df = mergeMatchupDATA(s25_regular, gameLogsTotal)
df.head()

,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,E_OFF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID_x,whos_favored,spread,total,team_is_favored,team_spread,OPP_BLOWOUT_RISK,TEAM_NAME,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,percentageFieldGoalsAttempted2pt,percentageFieldGoalsAttempted3pt,percentagePoints2pt,percentagePointsMidrange2pt,percentagePoints3pt,percentagePointsFastBreak,percentagePointsFreeThrow,percentagePointsOffTurnovers,percentagePointsPaint,percentageAssisted2pt,percentageUnassisted2pt,percentageAssisted3pt,percentageUnassisted3pt,percentageAssistedFGM,percentageUnassistedFGM,matchupFieldGoalsMade,matchupFieldGoalsAttempted,matchupThreePointersMade,matchupThreePointersAttempted,playerPoints,matchupMinutes,DEF_FG_PCT_ALLOWED,DEF_3PT_PCT_ALLOWED,PTS_ALLOWED_PER_MIN
0,0,Mike Conley,201144,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,5,2,4,1,7,0.143,0,5,0.000,3,3,1.0,2,2,1,0,3,-22,12.8,0.601,0.142857,G,NaN,72.8,130.4,-57.0,0.091,0.091,0.091,0.222,0.143,0.67,0.220,0.300,101.33,96.16,0.016,41,80.13,0.229,20.22,4.45,1.62,3,3,6,55,0,0,40,0,2,0.00,1,5,0.200,1,1,1.000,0.0,5.0,0.0,2.0,17.0,8.0,15.0,38.0,1.0,1.0,1.0,0,22024,0,1.5,223.5,1,-1.5,0,Minnesota Timberwolves,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,99.4,99.4,99.4,1610612747,102.1,109.0,105.1,112.2,110.0,42.0,95.0,0.442,46.0,22.0,7.0,8.0,7.0,0.286,0.714,0.400,0.0,0.000,0.00,0.600,0.000,0.400,0.000,1.000,0.0,0.0,0.000,1.000,4.0,10.0,0.0,4.0,8.0,7.92,0.400,0.0,1.01
1,1,Dalton Knecht,1642261,LAL vs. MIN,LAL,1610612747,MIN,1,22400062,2024-10-22,W,5,2,1,2,4,0.500,1,3,0.333,0,0,NaN,0,1,1,0,1,7,11.2,1.250,0.625000,NaN,NaN,128.7,112.3,13.8,0.000,0.063,0.031,0.118,0.625,2.00,0.122,0.625,102.12,100.36,0.070,34,83.63,0.128,15.78,4.46,1.25,0,1,1,18,0,0,12,0,0,0.00,2,4,0.500,0,0,0.000,0.0,0.0,2.0,2.0,2.0,11.0,3.0,16.0,0.0,1.0,0.0,0,22024,0,1.5,223.5,0,1.5,0,Los Angeles Lakers,240,42,95,0.442,5,30,0.167,21,25,0.840,15,31,46,22,7,8,7,22,110,7,99.4,99.4,99.4,1610612750,112.2,105.1,109.0,102.1,103.0,35.0,85.0,0.412,47.0,17.0,4.0,1.0,16.0,0.250,0.750,0.400,0.0,0.600,0.40,0.000,0.000,0.400,0.000,1.000,1.0,0.0,0.500,0.500,2.0,6.0,2.0,5.0,6.0,5.91,0.333,0.4,1.02
2,2,Jaden McDaniels,1630183,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,6,1,2,3,8,0.375,0,3,0.000,0,0,NaN,0,2,1,0,1,-8,11.9,0.750,0.375000,F,NaN,80.0,116.4,-26.7,0.000,0.105,0.059,0.143,0.375,1.00,0.243,0.375,95.76,90.00,-0.022,30,75.00,0.245,16.00,4.53,1.28,3,5,8,21,0,0,10,1,2,0.50,2,6,0.333,1,1,1.000,2.0,0.0,0.0,6.0,9.0,7.0,7.0,18.0,1.0,5.0,2.0,0,22024,0,1.5,223.5,1,-1.5,0,Minnesota Timberwolves,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,99.4,99.4,99.4,1610612747,102.1,109.0,105.1,112.2,110.0,42.0,95.0,0.442,46.0,22.0,7.0,8.0,7.0,0.625,0.375,1.000,0.0,0.000,0.00,0.000,0.333,1.000,0.667,0.333,0.0,0.0,0.667,0.333,3.0,6.0,1.0,2.0,9.0,6.41,0.500,0.5,1.40
3,3,Naz Reid,1629675,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,12,1,4,3,8,0.375,2,4,0.500,4,4,1.0,1,3,0,0,1,-6,17.3,1.230,0.500000,NaN,NaN,112.9,118.4,-9.0,0.034,0.107,0.070,0.056,0.500,1.00,0.172,0.615,100.77,97.46,0.074,53,81.

In [ ]:
# df.to_csv('../DATA/CSV_FILES/REGULAR_DATA/S25.csv', index=False)

In [73]:
from nba_api.stats.endpoints import leaguedashptdefend
categories = ['3 Pointers', '2 Pointers', 'Less Than 6Ft', 'Less Than 10Ft', 'Greater Than 15Ft']
data = []
for category in categories:
    league_df = leaguedashptdefend.LeagueDashPtDefend(
        per_mode_simple='PerGame',
        defense_category=category,
        season='2024-25',
        season_type_all_star='Regular Season'
    ).get_data_frames()[0]
    league_df[f'FREQ_{category}'] = league_df['FREQ']
    league_df[f'PLUS_MINUS_{category}'] = league_df['PLUSMINUS']
    data.append(league_df)

league_df = pd.concat(data)
player_stats = league_df.groupby([
    'CLOSE_DEF_PERSON_ID', 
    'PLAYER_NAME', 
    'PLAYER_LAST_TEAM_ID', 
    'PLAYER_LAST_TEAM_ABBREVIATION', 
    'PLAYER_POSITION', 
    'AGE', 
    'GP'
]).agg('max').reset_index()

player_stats = player_stats[['CLOSE_DEF_PERSON_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'PLAYER_POSITION', 'AGE','FREQ_3 Pointers', 'FG3M', 'FG3A', 'FG3_PCT', 'NS_FG3_PCT','PLUS_MINUS_3 Pointers', 'FREQ_2 Pointers', 'FG2M', 'FG2A', 'FG2_PCT', 'NS_FG2_PCT', 'PLUS_MINUS_2 Pointers', 'FREQ_Less Than 6Ft', 'FGM_LT_06', 'FGA_LT_06', 'LT_06_PCT', 'NS_LT_06_PCT', 'PLUS_MINUS_Less Than 6Ft', 'FREQ_Less Than 10Ft', 'FGM_LT_10', 'FGA_LT_10', 'LT_10_PCT', 'NS_LT_10_PCT', 'PLUS_MINUS_Less Than 10Ft', 'FREQ_Greater Than 15Ft', 'FGM_GT_15', 'FGA_GT_15', 'GT_15_PCT', 'NS_GT_15_PCT', 'PLUS_MINUS_Greater Than 15Ft']]
player_stats.rename(columns={'CLOSE_DEF_PERSON_ID': 'PLAYER_ID','FREQ_3 Pointers': 'FREQ_FG3', 'FREQ_2 Pointers': 'FREQ_FG2', 'FREQ_Less Than 6Ft': 'FREQ_LT_06', 'FREQ_Less Than 10Ft': 'FREQ_LT_10', 'FREQ_Greater Than 15Ft': 'FREQ_GT_15', 'PLUS_MINUS_3 Pointers': 'PLUS_MINUS_FG3','PLUS_MINUS_2 Pointers': 'PLUS_MINUS_FG2','PLUS_MINUS_Less Than 6Ft': 'PLUS_MINUS_LT_06', 'PLUS_MINUS_Less Than 10Ft': 'PLUS_MINUS_LT_10','PLUS_MINUS_Greater Than 15Ft': 'PLUS_MINUS_GT_15' }, inplace=True)
player_stats



,PLAYER_ID,PLAYER_NAME,PLAYER_LAST_TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,PLAYER_POSITION,AGE,FREQ_FG3,FG3M,FG3A,FG3_PCT,NS_FG3_PCT,PLUS_MINUS_FG3,FREQ_FG2,FG2M,FG2A,FG2_PCT,NS_FG2_PCT,PLUS_MINUS_FG2,FREQ_LT_06,FGM_LT_06,FGA_LT_06,LT_06_PCT,NS_LT_06_PCT,PLUS_MINUS_LT_06,FREQ_LT_10,FGM_LT_10,FGA_LT_10,LT_10_PCT,NS_LT_10_PCT,PLUS_MINUS_LT_10,FREQ_GT_15,FGM_GT_15,FGA_GT_15,GT_15_PCT,NS_GT_15_PCT,PLUS_MINUS_GT_15
0,2544,LeBron James,1610612747,LAL,F,40.0,0.470,1.77,5.21,0.340,0.356,-0.017,0.530,3.14,5.87,0.535,0.547,-0.011,0.311,2.23,3.44,0.647,0.628,0.019,0.410,2.66,4.54,0.585,0.586,-0.001,0.513,1.87,5.69,0.329,0.363,-0.033
1,101108,Chris Paul,1610612759,SAS,G,40.0,0.446,1.77,4.54,0.390,0.359,0.030,0.554,3.17,5.63,0.563,0.544,0.018,0.384,2.43,3.90,0.622,0.630,-0.008,0.448,2.71,4.56,0.594,0.586,0.008,0.496,1.95,5.05,0.386,0.365,0.022
2,200768,Kyle Lowry,1610612755,PHI,G,39.0,0.424,1.20,2.94,0.408,0.361,0.047,0.576,2.29,4.00,0.571,0.542,0.029,0.346,1.54,2.40,0.643,0.635,0.008,0.416,1.77,2.89,0.614,0.585,0.029,0.498,1.43,3.46,0.413,0.366,0.047
3,200782,P.J. Tucker,1610612752,NYK,F,40.0,0.545,2.33,4.00,0.583,0.328,0.256,0.455,2.33,3.33,0.700,0.505,0.195,0.318,1.67,2.33,0.714,0.592,0.123,0.364,1.67,2.67,0.625,0.514,0.111,0.591,2.67,4.33,0.615,0.336,0.279
4,201142,Kevin Durant,1610612756,PHX,F,36.0,0.487,2.15,6.03,0.356,0.362,-0.006,0.513,2.95,6.35,0.464,0.545,-0.080,0.293,2.03,3.63,0.560,0.628,-0.068,0.388,2.42,4.81,0.503,0.585,-0.082,0.546,2.34,6.76,0.346,0.366,-0.020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,1642461,Spencer Jones,1610612743,DEN,F,24.0,0.545,0.38,1.38,0.278,0.362,-0.084,0.455,0.38,1.15,0.333,0.503,-0.170,0.273,0.23,0.69,0.333,0.548,-0.215,0.333,0.31,0.85,0.364,0.525,-0.161,0.576,0.38,1.46,0.263,0.369,-0.106
564,1642484,RayJ Dennis,1610612754,IND,G,24.0,0.375,0.57,1.71,0.333,0.319,0.014,0.625,1.86,2.86,0.650,0.519,0.131,0.219,0.57,1.00,0.571,0.608,-0.037,0.438,1.14,2.00,0.571,0.551,0.021,0.438,0.86,2.00,0.429,0.322,0.107
565,1642502,Malevy Leons,1610612760,OKC,F,25.0,0.571,0.50,1.00,0.500,0.308,0.193,0.429,0.50,0.75,0.667,0.492,0.175,0.286,0.25,0.50,0.500,0.585,-0.085,0.429,0.50,0.75,0.667,0.535,0.131,0.571,0.50,1.00,0.500,0.319,0.181
566,1642505,Alex Ducas,1610612760,OKC,G,24.0,0.450,0.54,1.38,0.389,0.345,0.044,0.550,1.08,1.69,0.636,0.523,0.114,0.300,0.69,0.92,0.750,0.562,0.188,0.375,0.77,1.15,0.667,0.546,0.121,0.525,0.69,1.62,0.429,0.352,0.077


In [ ]:
from nba_api.stats.endpoints import boxscorescoringv3
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
from requests.exceptions import ReadTimeout, ConnectionError

def fetch_game_data(gameId, sleep_time=1.0):
    """Fetch data for a single game with error handling"""
    try:
        time.sleep(sleep_time)
        
        boxscore = boxscorescoringv3.BoxScoreScoringV3(
            game_id=f'00{gameId}',
            timeout=60
        )
        data_frames = boxscore.get_data_frames()
        if data_frames and len(data_frames) > 0 and not data_frames[0].empty:
            return data_frames[0], gameId, None
        else:
            return None, gameId, "No data available"
            
    except ReadTimeout:
        return None, gameId, "Timeout"
    except Exception as e:
        return None, gameId, str(e)

# Get unique game IDs
gameIds = s25_regular['GAME_ID'].unique()
games = []
failed_games = []

print(f"Processing {len(gameIds)} games using ThreadPoolExecutor...")

max_workers = 3 
sleep_time = 1.5  

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_game = {
        executor.submit(fetch_game_data, gameId, sleep_time): gameId 
        for gameId in gameIds
    }
    
    # Process completed tasks
    for i, future in enumerate(as_completed(future_to_game), 1):
        gameId = future_to_game[future]
        
        try:
            result, game_id, error = future.result()
            
            if result is not None:
                games.append(result)
                print(f"✓ Game {gameId} completed ({i}/{len(gameIds)})")
            else:
                failed_games.append((gameId, error))
                print(f"✗ Game {gameId} failed: {error} ({i}/{len(gameIds)})")
                
        except Exception as e:
            failed_games.append((gameId, str(e)))
            print(f"✗ Game {gameId} exception: {str(e)} ({i}/{len(gameIds)})")

print(f"\nCompleted!")
print(f"Successfully processed: {len(games)} games")
print(f"Failed games: {len(failed_games)}")

if failed_games:
    print(f"Failed games: {failed_games[:10]}...")

if games:
    games_df = pd.concat(games, ignore_index=True)
    print(f"\nFinal dataset shape: {games_df.shape}")
    games_df.head()
else:
    print("No games were successfully processed!")

In [ ]:
def mergeDATA(s19_data, boxscoringv3_data):
    s19_data = s19_data.copy()
    boxscoringv3_data = boxscoringv3_data.copy()
    
    if 'gameId' in boxscoringv3_data.columns:
        boxscoringv3_data['GAME_ID'] = boxscoringv3_data['gameId'].astype(int)
    
    if 'personId' in boxscoringv3_data.columns:
        boxscoringv3_data['PLAYER_ID'] = boxscoringv3_data['personId']
    
    merge_columns = [
        'GAME_ID', 'PLAYER_ID', 
        'percentageFieldGoalsAttempted2pt', 'percentageFieldGoalsAttempted3pt', 
        'percentagePoints2pt', 'percentagePointsMidrange2pt', 'percentagePoints3pt',
        'percentagePointsFastBreak', 'percentagePointsFreeThrow', 'percentagePointsOffTurnovers', 
        'percentagePointsPaint', 'percentageAssisted2pt', 'percentageUnassisted2pt',
        'percentageAssisted3pt', 'percentageUnassisted3pt', 'percentageAssistedFGM', 
        'percentageUnassistedFGM'
    ]
    
    available_columns = [col for col in merge_columns if col in boxscoringv3_data.columns]
    boxscoringv3_subset = boxscoringv3_data[available_columns].copy()
    merged_data = s19_data.merge(
        boxscoringv3_subset,
        on=['GAME_ID', 'PLAYER_ID'],
        how='left',
        suffixes=('', '_boxscore')
    )
    return merged_data

df = mergeDATA(s25_regular, games_df)
df.drop(columns=['Unnamed: 0', 'TEAM_SEASON_ID', 'TEAM_GAME_DATE', 'TEAM_MATCHUP', 'TEAM_WL', 'VIDEO_AVAILABLE' ], inplace=True)
df.head()

,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,E_OFF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID_x,whos_favored,spread,total,team_is_favored,team_spread,OPP_BLOWOUT_RISK,TEAM_SEASON_ID,TEAM_NAME,TEAM_GAME_DATE,TEAM_MATCHUP,TEAM_WL,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,VIDEO_AVAILABLE,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,percentageFieldGoalsAttempted2pt,percentageFieldGoalsAttempted3pt,percentagePoints2pt,percentagePointsMidrange2pt,percentagePoints3pt,percentagePointsFastBreak,percentagePointsFreeThrow,percentagePointsOffTurnovers,percentagePointsPaint,percentageAssisted2pt,percentageUnassisted2pt,percentageAssisted3pt,percentageUnassisted3pt,percentageAssistedFGM,percentageUnassistedFGM
0,0,Mike Conley,201144,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,5,2,4,1,7,0.143,0,5,0.000,3,3,1.0,2,2,1,0,3,-22,12.8,0.601,0.142857,G,NaN,72.8,130.4,-57.0,0.091,0.091,0.091,0.222,0.143,0.67,0.220,0.300,101.33,96.16,0.016,41,80.13,0.229,20.22,4.45,1.62,3,3,6,55,0,0,40,0,2,0.00,1,5,0.200,1,1,1.000,0.0,5.0,0.0,2.0,17.0,8.0,15.0,38.0,1.0,1.0,1.0,0,22024,0,1.5,223.5,1,-1.5,0,22024,Minnesota Timberwolves,2024-10-22,MIN @ LAL,L,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,1,99.4,99.4,99.4,1610612747,102.1,109.0,105.1,112.2,110.0,42.0,95.0,0.442,46.0,22.0,7.0,8.0,7.0,0.286,0.714,0.400,0.0,0.000,0.00,0.600,0.000,0.400,0.000,1.000,0.0,0.0,0.000,1.000
1,1,Dalton Knecht,1642261,LAL vs. MIN,LAL,1610612747,MIN,1,22400062,2024-10-22,W,5,2,1,2,4,0.500,1,3,0.333,0,0,NaN,0,1,1,0,1,7,11.2,1.250,0.625000,NaN,NaN,128.7,112.3,13.8,0.000,0.063,0.031,0.118,0.625,2.00,0.122,0.625,102.12,100.36,0.070,34,83.63,0.128,15.78,4.46,1.25,0,1,1,18,0,0,12,0,0,0.00,2,4,0.500,0,0,0.000,0.0,0.0,2.0,2.0,2.0,11.0,3.0,16.0,0.0,1.0,0.0,0,22024,0,1.5,223.5,0,1.5,0,22024,Los Angeles Lakers,2024-10-22,LAL vs. MIN,W,240,42,95,0.442,5,30,0.167,21,25,0.840,15,31,46,22,7,8,7,22,110,7,1,99.4,99.4,99.4,1610612750,112.2,105.1,109.0,102.1,103.0,35.0,85.0,0.412,47.0,17.0,4.0,1.0,16.0,0.250,0.750,0.400,0.0,0.600,0.40,0.000,0.000,0.400,0.000,1.000,1.0,0.0,0.500,0.500
2,2,Jaden McDaniels,1630183,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,6,1,2,3,8,0.375,0,3,0.000,0,0,NaN,0,2,1,0,1,-8,11.9,0.750,0.375000,F,NaN,80.0,116.4,-26.7,0.000,0.105,0.059,0.143,0.375,1.00,0.243,0.375,95.76,90.00,-0.022,30,75.00,0.245,16.00,4.53,1.28,3,5,8,21,0,0,10,1,2,0.50,2,6,0.333,1,1,1.000,2.0,0.0,0.0,6.0,9.0,7.0,7.0,18.0,1.0,5.0,2.0,0,22024,0,1.5,223.5,1,-1.5,0,22024,Minnesota Timberwolves,2024-10-22,MIN @ LAL,L,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,1,99.4,99.4,99.4,1610612747,102.1,109.0,105.1,112.2,110.0,42.0,95.0,0.442,46.0,22.0,7.0,8.0,7.0,0.625,0.375,1.000,0.0,0.000,0.00,0.000,0.333,1.000,0.667,0.333,0.0,0.0,0.667,0.333
3,3,Naz Reid,1629675,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,12,1,4,3,8,0.375,2,4,0.500,4,4,1.0,1,3,0,0,1,-6,17.3,1.230,0.500000,NaN,NaN,112.9,118.4,-9.0,0.034,0.107,0.070,0.056,0.500,1.00,0.172,0.615,100.77,97.46,0.074,53,81.21,0.174,26.35,4.22,2.00,3,9,12,34,0,0,23,1,4,0.25,2,4,0.500,5,8,0.625,2.0,2.0,3.0,2.0,8.0,13.0,8.0,46.0,0.0,3.0,2.0,0,22024,0,1.5,223.5,1,-1.5,0,2202

## Fetches Player Gamelogs

In [2]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2018-19', 
#     season_type='Regular Season', 
#     sleep_time=1.5, 
#     max_workers=5,
#     batch_limit=100,
#     complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S19.csv'
# )
# data.head()

## Merge team data into player logs

In [4]:
# # Drop all team-related columns to re-merge with correct pace calculations
# data = s19_regular

# columns_to_drop = [
#     'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0',
#     # Pace columns (incorrect calculations)
#     'TEAM_PACE', 'GAME_PACE', 'OPP_PACE',
    
#     # Rating columns (may need recalculation with correct pace)
#     'TEAM_OFF_RATING', 'TEAM_DEF_RATING',
#     'OPP_OFF_RATING', 'OPP_DEF_RATING',
    
#     # Other team stats that came from the merge
#     'TEAM_PTS', 'TEAM_FGM', 'TEAM_FGA', 'TEAM_FG_PCT',
#     'TEAM_FG3M', 'TEAM_FG3A', 'TEAM_FG3_PCT',
#     'TEAM_FTM', 'TEAM_FTA', 'TEAM_FT_PCT', 
#     'TEAM_OREB', 'TEAM_DREB', 'TEAM_REB',
#     'TEAM_AST', 'TEAM_STL', 'TEAM_BLK', 'TEAM_TOV',
#     'TEAM_PF', 'TEAM_PLUS_MINUS',
    
#     # Opponent stats
#     'OPP_TEAM_ID', 'OPP_PTS', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
#     'OPP_REB', 'OPP_AST', 'OPP_STL', 'OPP_BLK', 'OPP_TOV'
    
#     'TEAM_SEASON_ID_x',	'OPP_TOV_x', 'TEAM_SEASON_ID_y', 'OPP_TOV_y', 'TEAM_SEASON_ID', 'TEAM_NAME', 'TEAM_GAME_DATE', 'TEAM_MATCHUP', 'TEAM_WL', 'TEAM_MIN', 'VIDEO_AVAILABLE'
# ]

# # Drop columns that exist in the dataframe
# existing_cols = [col for col in columns_to_drop if col in data.columns]
# data = data.drop(columns=existing_cols)

# teamlogs = mergeTeamtoPlayer(data, season='2018-19', season_type='Regular Season')
# teamlogs


In [5]:
# teamlogs.to_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
# teamlogs

## Assign features for regular season data

In [6]:
star_players_by_year = {
    2019:[ "Giannis Antetokounmpo",
    "LeBron James",
    "Anthony Davis",
    "James Harden",
    "Luka Dončić",
    "Kawhi Leonard",
    "Pascal Siakam",
    "Nikola Jokić",
    "Damian Lillard",
    "Chris Paul",
    "Jayson Tatum",
    "Jimmy Butler",
    "Rudy Gobert",
    "Ben Simmons",
    "Russell Westbrook"],
    2020:[
    "Giannis Antetokounmpo",
    "Kawhi Leonard", 
    "Nikola Jokić",
    "Stephen Curry",
    "Luka Dončić",
    "Julius Randle",
    "LeBron James",
    "Joel Embiid",
    "Chris Paul",
    "Damian Lillard",
    "Jimmy Butler",
    "Paul George",
    "Rudy Gobert",
    "Bradley Beal",
    "Kyrie Irving"
    ],
    2021: [
        "Giannis Antetokounmpo", "Kawhi Leonard", "Nikola Jokić", "Stephen Curry", "Luka Dončić",
        "Julius Randle", "LeBron James", "Joel Embiid", "Damian Lillard", "Chris Paul",
        "Jimmy Butler", "Paul George", "Rudy Gobert", "Bradley Beal", "Kyrie Irving",
        "Devin Booker", "Mike Conley", "James Harden", "Zach LaVine", "Donovan Mitchell",
        "Nikola Vucevic", "Anthony Davis"
    ],
    2022: [
        "Giannis Antetokounmpo", "Luka Dončić", "Jayson Tatum", "Nikola Jokić", "Devin Booker",
        "Ja Morant", "Stephen Curry", "DeMar DeRozan", "Kevin Durant", "Joel Embiid",
        "LeBron James", "Chris Paul", "Trae Young", "Pascal Siakam", "Karl-Anthony Towns",
        "Andrew Wiggins", "Donovan Mitchell", "Rudy Gobert", "Zach LaVine", "Khris Middleton",
        "Jimmy Butler", "Darius Garland", "Fred VanVleet", "LaMelo Ball"
    ],
    2023: [
        "Giannis Antetokounmpo", "Jayson Tatum", "Joel Embiid", "Shai Gilgeous-Alexander", "Luka Dončić",
        "Jaylen Brown", "Jimmy Butler", "Nikola Jokić", "Stephen Curry", "Donovan Mitchell",
        "LeBron James", "Julius Randle", "Domantas Sabonis", "De'Aaron Fox", "Damian Lillard",
        "Kyrie Irving", "Zion Williamson", "Kevin Durant", "Ja Morant", "DeMar DeRozan",
        "Tyrese Haliburton", "Jrue Holiday", "Bam Adebayo", "Jaren Jackson Jr.", "Paul George",
        "Pascal Siakam", "Anthony Edwards"
    ],
    2024: [
        "Shai Gilgeous-Alexander", "Luka Dončić", "Jayson Tatum", "Giannis Antetokounmpo", "Nikola Jokić",
        "Jalen Brunson", "Anthony Edwards", "Kawhi Leonard", "Kevin Durant", "Anthony Davis",
        "Stephen Curry", "Devin Booker", "LeBron James", "Domantas Sabonis", "Bam Adebayo",
        "Tyrese Haliburton", "Damian Lillard", "Karl-Anthony Towns", "Jaylen Brown",
        "Trae Young", "Paolo Banchero", "Scottie Barnes", 'Joel Embiid'
    ],
    2025: [
        'Shai Gilgeous-Alexander', 'Nikola Jokić', 'Giannis Antetokounmpo', 'Jayson Tatum', 'Donovan Mitchell',
        'Anthony Edwards', 'LeBron James', 'Stephen Curry', 'Evan Mobley', 'Jalen Brunson',
        'Cade Cunningham', 'Karl-Anthony Towns', 'Tyrese Haliburton', 'Jalen Williams', 'James Harden',
        'Darius Garland', 'Damian Lillard', 'Anthony Davis', 'Kyrie Irving', 'Jaylen Brown', 'Tyler Herro', 'Jaren Jackson Jr.', 
        'Pascal Siakam', 'Victor Wembanyama', 'Alperen Sengun', 'Trae Young', 'LaMelo Ball', 'Devin Booker', 'Joel Embiid', 'Luka Dončić'
]}

In [7]:
def process_season_features(season_df, prop_type, year, star_players):
    df = season_df.copy()
    df = sort_data_for_features(df)
    df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
    # df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)
    df['TEAM_IMPLIED_PTS_FAV'] = ((df['total'] + df['team_spread']) / 2).round(1)
    df['TEAM_IMPLIED_PTS_UND'] = ((df['total'] - df['team_spread']) / 2).round(1)
    
    # Add position data
    cache_file = os.path.join(feature_path, 'DATA', 'TOOLS', 'playerInfo.csv')
    df = assign_position_with_cache(
        df, 
        cache_file=cache_file,
        max_workers=4, 
        delay_between_requests=1.5
    )
    # Basic features to track rest and travel
    df = add_rest_day_features(df)
    
    # Prop-specific features
    df = rollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,7])
    df = statAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION', stat_line=prop_type)
    df = HomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
    df = getPlayerAvgToDateVectorized(df)
    df = addLagFeatures(df, stat_lines=['PTS', 'MIN', 'FGA', 'FTA', 'FG3A', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'USG_PCT', 'TS_PCT',
                        'EFG_PCT','POSS', 'TCHS','AST', 'REB', 'TOV'])
    # df = add_volatility_features(df, windows=[3,5,7,15,40])
    # df = add_performance_volatility_categories(df)
    # df = add_recent_form_volatility(df)
    df = process_star_players_data(df, star_players, min_minutes=20)
    df = add_performance_without_stars_columns(df, min_games=1)
    df = teamUsualStarters(df)
    df = oppTeamUsualStarters(df)
    # df = add_all_defensive_features(df, all_defensive_players, year)
    df = teamContext(df)
    df = assign_opponent_team_stats_dict(df)
    df = expectedPace(df)
    
    # Clean up any unwanted columns
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    if 'Unnamed: 0.1' in df.columns:
        df.drop(columns=['Unnamed: 0.1'], inplace=True)
    return df

prop_types = ['PTS']
data = [s19_regular,s20_regular, s21_regular,s22_regular,s23_regular,s24_regular,s25_regular]
seasons = [2019,2020,2021,2022,2023,2024,2025]

# Pre-define output directory once
output_dir = os.path.join(feature_path, 'DATA', 'CSV_FILES', 'TRAIN_DATA')
os.makedirs(output_dir, exist_ok=True)

# Process each season sequentially but with optimized operations
for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year,
        star_players=set(star_players_by_year[year]),
        # all_defensive_players=set(all_defensive_players[year])
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2019...
Loading position cache...
Loaded 992 players from cache
Found 530 unique players, 159 need to be fetched
Fetching 159 new players using 4 threads...
Fetched PLAYER_ID 1717... (1/159)
Fetched PLAYER_ID 1713... (2/159)
Fetched PLAYER_ID 2037... (3/159)
Fetched PLAYER_ID 2199... (4/159)
Fetched PLAYER_ID 2200... (5/159)
Fetched PLAYER_ID 2225... (6/159)
Fetched PLAYER_ID 2403... (7/159)
Fetched PLAYER_ID 2548... (8/159)
Fetched PLAYER_ID 2585... (9/159)
Fetched PLAYER_ID 2594... (10/159)
Fetched PLAYER_ID 2733... (11/159)
Fetched PLAYER_ID 2734... (12/159)
Fetched PLAYER_ID 2736... (13/159)
Fetched PLAYER_ID 101107... (14/159)
Fetched PLAYER_ID 2747... (15/159)
Fetched PLAYER_ID 101106... (16/159)
Fetched PLAYER_ID 101109... (17/159)
Fetched PLAYER_ID 101112... (18/159)
Fetched PLAYER_ID 101123... (19/159)
Fetched PLAYER_ID 101133... (20/159)
Fetched PLAYER_ID 101161... (21/159)
Fetched PLAYER_ID 101162... (22/159)
Fetched PLAYER_ID 101181... (23/159)
Fetched PLAYE

/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2019
Processing year 2020...
Loading position cache...
Loaded 1151 players from cache
Found 529 unique players, 24 need to be fetched
Fetching 24 new players using 4 threads...
Fetched PLAYER_ID 1626187... (1/24)
Fetched PLAYER_ID 1627784... (2/24)
Fetched PLAYER_ID 202951... (3/24)
Fetched PLAYER_ID 203705... (4/24)
Fetched PLAYER_ID 1628430... (5/24)
Fetched PLAYER_ID 1627982... (6/24)
Fetched PLAYER_ID 1628450... (7/24)
Fetched PLAYER_ID 1628499... (8/24)
Fetched PLAYER_ID 1628987... (9/24)
Fetched PLAYER_ID 1628985... (10/24)
Fetched PLAYER_ID 1629044... (11/24)
Fetched PLAYER_ID 1629065... (12/24)
Fetched PLAYER_ID 1629598... (13/24)
Fetched PLAYER_ID 1629608... (14/24)
Fetched PLAYER_ID 1629625... (15/24)
Fetched PLAYER_ID 1629621... (16/24)
Fetched PLAYER_ID 1629668... (17/24)
Fetched PLAYER_ID 1629724... (18/24)
Fetched PLAYER_ID 1629729... (19/24)
Fetched PLAYER_ID 1629734... (20/24)
Fetched PLAYER_ID 1629739... (21/24)
Fetched PLAYER_ID 1629741... (22/24)
Fetched PL

/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2020
Processing year 2021...
Loading position cache...
Loaded 1175 players from cache
Found 540 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


...

Completed 2023
Processing year 2024...
Loading position cache...
Loaded 1175 players from cache
Found 572 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2024
Processing year 2025...
Loading position cache...
Loaded 1175 players from cache
Found 569 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2025


In [8]:
# df = s21

# df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
# df['team_is_favored'] = df['team_is_favored'].astype(int)
# df['whos_favored'] = df['whos_favored'].apply(lambda x: 1 if x == 'home' else 0)
# df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)

# # Add position data
# cache_file = os.path.join(feature_path, 'DATA', 'TOOLS', 'playerInfo.csv')
# df = assign_position_with_cache(
#     df, 
#     cache_file=cache_file,
#     max_workers=4, 
#     delay_between_requests=1.5
# )
# df = add_rest_day_features(df)
# df = add_minutes_trend_features(df)
# df = add_lineup_cohesion(df)
# df = add_rotation_stability(df)
# df = add_usage_shift(df)
# df = MINrollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,7])
# df = MINLagFeatures(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', stat_line='MIN')
# df = getPlayerMINAvgToDate(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
# df = MINHomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
# df = MINAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION', stat_line='MIN')
# df = assign_opponent_team_stats_dict(df)
# df = teamContext(df)
# df.to_csv('../DATA/CSV_FILES/TRAIN_DATA/MIN_TRAIN_21.csv', index=False)

In [9]:
def merge_betting_data(player_df, betting_df, team_dict):
    """
    Merge betting data (spread, total, who's favored) into player dataset
    """
    df = player_df.copy()
    odds = betting_df.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    odds['date'] = pd.to_datetime(odds['date'])
    
    # Convert betting data team abbreviations to uppercase using team_dict
    odds['away_upper'] = odds['away'].map(team_dict)
    odds['home_upper'] = odds['home'].map(team_dict)
    
    # First, create a unique identifier for each game in odds data
    odds['game_key_home'] = odds['date'].astype(str) + '_' + odds['home_upper'] + '_' + odds['away_upper']
    odds['game_key_away'] = odds['date'].astype(str) + '_' + odds['away_upper'] + '_' + odds['home_upper']
    
    df['game_key'] = df['GAME_DATE'].astype(str) + '_' + df['TEAM_ABBREVIATION'] + '_' + df['OPP_ABBREVIATION']
    home_merge = df.merge(
        odds[['game_key_home', 'whos_favored', 'spread', 'total']].rename(columns={'game_key_home': 'game_key'}),
        on='game_key',
        how='left',
        suffixes=('', '_home')
    )
    away_merge = df.merge(
        odds[['game_key_away', 'whos_favored', 'spread', 'total']].rename(columns={'game_key_away': 'game_key'}),
        on='game_key', 
        how='left',
        suffixes=('', '_away')
    )
    df['whos_favored'] = home_merge['whos_favored'].fillna(away_merge['whos_favored'])
    df['spread'] = home_merge['spread'].fillna(away_merge['spread']).round(2)
    df['total'] = home_merge['total'].fillna(away_merge['total']).round(2)
    df['team_is_favored'] = ((df['whos_favored'] == 'home') & (df['HOME_GAME'] == 1)) | \
                           ((df['whos_favored'] == 'away') & (df['HOME_GAME'] == 0))
    df['team_spread'] = df.apply(lambda row: 
        round(row['spread'] if row['HOME_GAME'] == 1 else -row['spread'], 2), axis=1)
    df.drop('game_key', axis=1, inplace=True)
    return df

team_dict = {
    'min': 'MIN', 
    'bos': 'BOS', 
    'bkn': 'BKN', 
    'ny': 'NYK', 
    'phi': 'PHI', 
    'tor': 'TOR', 
    'chi': 'CHI', 
    'cle': 'CLE', 
    'det': 'DET', 
    'ind': 'IND', 
    'mia': 'MIA', 
    'atl': 'ATL', 
    'cha': 'CHA', 
    'was': 'WAS',
    'wsh': 'WAS',
    'orl': 'ORL', 
    'mil': 'MIL', 
    'chh': 'CHH', 
    'dal': 'DAL', 
    'hou': 'HOU',
    'lac': 'LAC',
    'lal': 'LAL',
    'sac': 'SAC',
    'por': 'POR',
    'uta': 'UTA',
    'utah': 'UTA', 
    'den': 'DEN',
    'okc': 'OKC',
    'mem': 'MEM',
    'no': 'NOP',
    'sa': 'SAS',    
    'gs': 'GSW',
    'phx': 'PHX',  
}

In [10]:
bettingData = pd.read_csv('../DATA/CSV_FILES/bettingData.csv')
df = merge_betting_data(s20_regular, bettingData, team_dict)
df

,Unnamed: 0.1,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,OFF_RATING,E_OFF_RATING,DEF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID,TEAM_NAME,TEAM_GAME_DATE,TEAM_MATCHUP,TEAM_WL,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,VIDEO_AVAILABLE,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,whos_favored,spread,total,team_is_favored,team_spread
0,0,0,Derrick Favors,202324,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,6,2,7,3,6,0.500,0,0,NaN,0,0,NaN,1,6,0,1,1,-12,19.4,1.000,0.500000,C,NaN,108.2,108.2,132.7,132.7,-24.5,0.048,0.261,0.159,0.118,0.500,2.0,0.137,0.500,113.31,113.31,0.056,49,94.43,0.137,20.75,4.60,1.75,4,8,10,43,0,0,35,3,3,1.000,0,3,0.000,7,8,0.875,0.0,2.0,0.0,6.0,12.0,11.0,7.0,32.0,0.0,5.0,0.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
1,1,1,Brandon Ingram,1627742,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,22,5,5,8,19,0.421,2,5,0.400,4,4,1.0,0,5,1,2,2,-19,42.5,1.060,0.473684,F,NaN,102.6,102.6,122.5,122.5,-19.9,0.000,0.125,0.068,0.227,0.474,2.5,0.272,0.530,107.35,107.35,0.112,77,89.46,0.272,35.10,4.33,2.78,1,9,9,69,2,0,46,5,9,0.556,3,10,0.300,4,8,0.500,3.0,0.0,3.0,8.0,16.0,14.0,14.0,44.0,0.0,4.0,6.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
2,2,2,Josh Hart,1628404,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,15,1,10,4,9,0.444,3,5,0.600,4,4,1.0,4,6,0,1,1,-1,30.5,1.394,0.611111,NaN,NaN,105.5,105.5,101.7,101.7,3.7,0.111,0.167,0.139,0.067,0.611,1.0,0.174,0.697,96.28,96.28,0.213,55,80.24,0.174,28.17,4.31,2.22,5,8,13,42,0,0,27,2,5,0.400,2,4,0.500,2,4,0.500,0.0,4.0,2.0,2.0,10.0,5.0,12.0,22.0,1.0,4.0,4.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
3,3,3,Lou Williams,101150,LAC vs. LAL,LAC,1610612746,LAL,1,21900002,2019-10-22,W,21,7,5,8,14,0.571,1,4,0.250,4,4,1.0,1,4,1,0,2,13,38.5,1.332,0.607143,NaN,NaN,121.6,121.6,102.7,102.7,19.0,0.029,0.098,0.066,0.280,0.607,3.5,0.214,0.666,97.38,97.38,0.195,74,81.15,0.214,36.72,3.76,2.46,1,5,6,89,0,2,62,1,2,0.500,7,12,0.583,0,1,0.000,8.0,2.0,5.0,6.0,9.0,6.0,5.0,30.0,1.0,0.0,9.0,0,22019,LA Clippers,2019-10-22,LAC vs. LAL,W,240,42,81,0.519,11,31,0.355,17,24,0.708,11,34,45,24,8,5,14,25,112,10,1,97.4,97.4,97.4,1610612747,118.4,107.9,111.7,101.8,102.0,37.0,85.0,0.435,41.0,20.0,4.0,7.0,15.0,away,3.5,224.0,False,3.5
4,4,4,Patrick Beverley,201976,LAC vs. LAL,LAC,1610612746,LAL,1,21900002,2019-10-22,W,2,6,10,1,7,0.143,0,5,0.000,0,0,NaN,2,8,0,1,2,13,24.0,0.286,0.142857,G,NaN,120.0,120.0,100.0,100.0,20.0,0.074,0.276,0.179,0.207,0.143,3.0,0.122,0.143,99.51,99.51,0.048,65,82.93,0.122,31.35,4.24,2.37,4,11,13,62,2,0,51,1,2,0.500,0,5,0.000,1,2,0.500,0.0,0.0